# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — the cohort, the target and the frozen baseline, rebuilt here

The building-baselines skill requires the baseline to appear *in the same table as the model,
computed in the same notebook run*. Citing ML-07's number would not satisfy that, so everything is
rebuilt: the cohort, the `asinh` target on its days 30–90 baseline, and the frozen gate.

Nothing here is new. If any figure disagrees with ML-07, that is a reproducibility failure and the
comparison below is void.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy scipy scikit-learn matplotlib python-dotenv

import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

SEED = 8
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | scikit-learn {sklearn.__version__}")

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

REPO = "FlyRank/internship-warehouse"
MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
daily_files = [hf_hub_download(repo_id=REPO, repo_type="dataset",
               filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
               for m in MONTHS]
dim_file = hf_hub_download(repo_id=REPO, repo_type="dataset",
                           filename="dim_content.parquet", token=token)
con = duckdb.connect()
REL = "read_parquet([" + ", ".join(f"'{f}'" for f in daily_files) + "])"
D1 = "2026-03-31"

q = f"""
WITH prior AS (
  SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
    SUM(gsc_impressions) AS impr_90d,
    SUM(gsc_clicks) AS clicks_90d,
    COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
    SUM(gsc_sum_position) AS sum_position_90d,
    SUM(gsc_impressions) FILTER (
        WHERE report_date < DATE '{D1}' - INTERVAL 30 DAY) AS older60_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D1}' - INTERVAL 30 DAY) AS recent30_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D1}' - INTERVAL 60 DAY
          AND report_date <  DATE '{D1}' - INTERVAL 30 DAY) AS base30_impr
  FROM {REL}
  WHERE report_date >= DATE '{D1}' - INTERVAL 90 DAY AND report_date < DATE '{D1}'
  GROUP BY content_hash_id HAVING SUM(gsc_impressions) > 0),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS future_impr FROM {REL}
  WHERE report_date >= DATE '{D1}' AND report_date < DATE '{D1}' + INTERVAL 30 DAY
  GROUP BY content_hash_id)
SELECT p.*, COALESCE(f.future_impr, 0) AS future_impr
FROM prior p LEFT JOIN future f USING (content_hash_id)
ORDER BY p.content_hash_id"""

df = con.sql(q).df()
dim = con.sql(f"""SELECT content_hash_id, content_created_date,
                         content_type, main_intent, competition_level,
                         search_volume, cpc, competition, backlinks,
                         keyword_char_count, keyword_token_count,
                         url_char_count, category_count,
                         keyword_created_date, last_optimized_date,
                         optimization_eligible_date
                  FROM read_parquet('{dim_file}')""").df()
df = df.merge(dim, on="content_hash_id", how="left")
for c in ["older60_impr", "recent30_impr", "base30_impr"]:
    df[c] = df[c].fillna(0)

DECISION = pd.Timestamp(D1)
df["baseline_daily"] = df["older60_impr"] / 60
df["recent_daily"] = df["recent30_impr"] / 30
df["future_daily"] = df["future_impr"] / 30
df["target"] = np.arcsinh(df["future_daily"]) - np.arcsinh(df["baseline_daily"])
df["declined"] = df["target"] < 0
df["avg_position"] = df["sum_position_90d"] / df["impr_90d"].replace(0, np.nan) + 1
df["ctr"] = df["clicks_90d"] / df["impr_90d"].replace(0, np.nan) * 100
df["log_impr_90d"] = np.log1p(df["impr_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["days_with_impressions"] = df["days_with_impressions"].fillna(0)
df["content_age_days"] = (DECISION - pd.to_datetime(df["content_created_date"])).dt.days
df["slip"] = np.where(df["baseline_daily"] > 0,
                      (df["baseline_daily"] - df["recent_daily"]) / df["baseline_daily"], np.nan)
df["peak_ratio"] = np.where(df["impr_90d"] > 0,
                            df["recent_daily"] / (df["impr_90d"] / 90), np.nan)
df["prior_trend"] = np.where(df["base30_impr"] > 0,
                             (df["recent30_impr"] - df["base30_impr"]) / df["base30_impr"], np.nan)

# --- the frozen ML-07 gate, verbatim
MIN_AGE_DAYS = 180
client_median_impr = df.groupby("client_hash_id")["impr_90d"].transform("median")
gate = ((df["baseline_daily"] > 0)
        & (df["content_age_days"] >= MIN_AGE_DAYS)
        & (df["impr_90d"] >= client_median_impr)
        & (df["slip"].fillna(0) <= 0.5))
df["in_gate"] = gate
pool = df[gate].copy()

print(f"cohort {len(df):,} pages | {df['client_hash_id'].nunique()} clients")
print(f"gated pool {len(pool):,} pages | {pool['client_hash_id'].nunique()} clients")
print(f"cohort decline rate {df['declined'].mean():.4f} | pool {pool['declined'].mean():.4f}")
print()
print("ML-07 reported: cohort 202,073 / 53 clients, pool 46,061, "
      "cohort rate 0.4228, pool rate 0.5128")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python 3.13.14 | pandas 3.0.1 | numpy 2.4.4 | scikit-learn 1.9.0


cohort 202,073 pages | 53 clients
gated pool 46,061 pages | 30 clients
cohort decline rate 0.4228 | pool 0.5128

ML-07 reported: cohort 202,073 / 53 clients, pool 46,061, cohort rate 0.4228, pool rate 0.5128


## 1. Method choice and why

**The question is "which pages first?", so the output has to be a score, not a label.** The
training-honest-models skill routes ranking questions to a model's continuous output evaluated at
`precision@K` — that is what a review queue consumes.

**Two shapes fit, and the choice is not obvious, so both get built:**

| approach | what it predicts | why it might win |
|---|---|---|
| **regressor** on the continuous `target` | how much the page moves | the target was rebuilt continuous precisely so magnitude survives; a 95% collapse and a 21% dip should not rank equally |
| **classifier** on `declined` | probability the page falls | matches the skill's default for ranking, and optimises the thing `precision@K` actually measures |

**Simple before strong, per the skill.** Ridge and logistic regression first — readable coefficients,
and a linear model that already works tells you the problem is easy. Then a gradient-boosted tree,
which handles the structural `NaN`s natively rather than by imputation. ML-06 Test 6 showed
`fillna` on a structurally-missing column stamps the content type into the feature, so no imputation
happens anywhere in this notebook.

**Two feature sets, because the strong features are the suspicious ones.**

| set | features | rationale |
|---|---|---|
| `SAFE` | `content_age_days`, `impr_90d`, `avg_position` | universally available, and none shares arithmetic with the target |
| `FULL` | `SAFE` + `prior_trend`, `peak_ratio` | correlates far more strongly (+0.54, +0.51) — and carries a residual artefact of about +0.11 |

Reporting only `FULL` would repeat ML-06's mistake three assignments later. The gap between the two
is the honest measure of what the trend features add.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

**`GroupShuffleSplit` on `client_hash_id`, ten seeds, `test_size=0.2` — identical to ML-07 section 6.**

Whole clients are held out, never individual pages. ML-05 measured what happens otherwise: the same
features on a random row split score **+0.091 AUC higher**, on every one of ten seeds, and that gap is
memorised client structure rather than skill.

**Three things must match the baseline or the comparison is meaningless**, and all three are fixed
here rather than chosen later: the same ten seeds, the same `test_size`, and the same gated pool. A
model ranking the full cohort would post a different number by answering a different question.

**Why ten seeds and not one.** ML-07 measured the pool's own decline rate swinging from **0.3180 to
0.7435** depending which clients land in the test set. A single split cannot distinguish a better
model from a luckier partition.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

Every row below is scored on the **same ten `GroupShuffleSplit` seeds**, the **same gated pool**, and
the **same pooled `precision@100`** — computed in this notebook run, not carried over.

**Ranking direction matters and is easy to get backwards.** A regressor predicts the target, where
*more negative* means a bigger fall, so it ranks **ascending**. A classifier predicts P(declined), so
it ranks **descending**. Getting this wrong produces a model that looks catastrophically bad, which is
a useful thing to be able to recognise.

**The two baseline rows are the bar**: the frozen gate with pages in random order, and the frozen
ML-07 rule ordering by age. ML-07 measured the second as *worse* than the first.

In [4]:
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

SAFE = ["content_age_days", "impr_90d", "avg_position"]
FULL = SAFE + ["prior_trend", "peak_ratio"]
K = 100

print("missing values inside the gated pool:")
for c in FULL:
    n = pool[c].isna().sum()
    print(f"  {c:18s} {n:,} ({n / len(pool):.2%})")

# One evaluation pool for every model, so the comparison is like-for-like.
# No imputation anywhere: rows missing a FULL feature are dropped once, up
# front, rather than filled with a number that would encode which client or
# content type they came from (ML-06 Test 6).
ev = pool.dropna(subset=FULL + ["target"]).copy()
print()
print(f"evaluation pool: {len(ev):,} of {len(pool):,} pool pages "
      f"({len(ev) / len(pool):.1%}), {ev['client_hash_id'].nunique()} clients")
print(f"decline rate on the evaluation pool: {ev['declined'].mean():.4f}")


def precision_at_k_per_client(frame, score, label="declined", k=K, ascending=False):
    hits = picks = 0
    for _, g in frame.groupby("client_hash_id"):
        g = g.sort_values(score, ascending=ascending).head(min(k, len(g)))
        hits += int(g[label].sum())
        picks += len(g)
    return hits / picks if picks else np.nan


def build(name, feats, kind):
    if kind == "ridge":
        return name, feats, "reg", lambda: Ridge(alpha=1.0, random_state=None)
    if kind == "logistic":
        return name, feats, "clf", lambda: LogisticRegression(max_iter=2000, random_state=SEED)
    if kind == "gbm_reg":
        return name, feats, "reg", lambda: HistGradientBoostingRegressor(random_state=SEED)
    return name, feats, "clf", lambda: HistGradientBoostingClassifier(random_state=SEED)


MODELS = [build(f"{k} {tag}", f, k)
          for k in ["ridge", "logistic", "gbm_reg", "gbm_clf"]
          for tag, f in [("SAFE", SAFE), ("FULL", FULL)]]

rng = np.random.default_rng(SEED)
rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]

    rows.append({"seed": seed, "model": "baseline  gate + random order",
                 "p_at_100": precision_at_k_per_client(
                     test.assign(z=rng.random(len(test))), "z")})
    rows.append({"seed": seed, "model": "baseline  ML-07 rule (age order)",
                 "p_at_100": precision_at_k_per_client(test, "content_age_days")})

    for name, feats, kind, make in MODELS:
        Xtr, Xte = train[feats], test[feats]
        if name.startswith(("ridge", "logistic")):
            sc = StandardScaler().fit(Xtr)
            Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
        m = make()
        if kind == "reg":
            m.fit(Xtr, train["target"])
            score, asc = m.predict(Xte), True
        else:
            m.fit(Xtr, train["declined"])
            score, asc = m.predict_proba(Xte)[:, 1], False
        rows.append({"seed": seed, "model": name,
                     "p_at_100": precision_at_k_per_client(
                         test.assign(s=score), "s", ascending=asc)})

res = pd.DataFrame(rows)
tbl = (res.groupby("model")["p_at_100"]
       .agg(mean="mean", min="min", max="max")
       .sort_values("mean", ascending=False).round(4))
rand = tbl.loc["baseline  gate + random order", "mean"]
tbl["vs random"] = (tbl["mean"] - rand).round(4)
tbl["beats random"] = [
    int((res[res.model == m].set_index("seed")["p_at_100"] >
         res[res.model == "baseline  gate + random order"].set_index("seed")["p_at_100"]).sum())
    for m in tbl.index]
print()
print(f"base rate on the evaluation pool: {ev['declined'].mean():.4f}")
print(tbl.to_string())

missing values inside the gated pool:
  content_age_days   0 (0.00%)
  impr_90d           0 (0.00%)
  avg_position       0 (0.00%)
  prior_trend        966 (2.10%)
  peak_ratio         0 (0.00%)

evaluation pool: 45,095 of 46,061 pool pages (97.9%), 30 clients
decline rate on the evaluation pool: 0.5131



base rate on the evaluation pool: 0.5131
                                    mean     min     max  vs random  beats random
model                                                                            
ridge FULL                        0.7563  0.5141  0.8430     0.2436            10
logistic FULL                     0.7500  0.5211  0.8340     0.2373            10
gbm_clf FULL                      0.7298  0.5141  0.8309     0.2171            10
gbm_reg FULL                      0.7216  0.4507  0.8333     0.2089            10
gbm_clf SAFE                      0.5526  0.3423  0.7184     0.0399             7
logistic SAFE                     0.5466  0.3308  0.6812     0.0339             7
ridge SAFE                        0.5421  0.3423  0.6933     0.0294             7
gbm_reg SAFE                      0.5245  0.3169  0.6729     0.0118             6
baseline  gate + random order     0.5127  0.3192  0.6429     0.0000             0
baseline  ML-07 rule (age order)  0.4882  0.3099  0.6751

### The result that has to be tested before it is believed

`FULL` beats random by **+0.21 to +0.24 on ten of ten seeds**. `SAFE` beats it by **+0.01 to +0.04**.
Everything separating them is `prior_trend` and `peak_ratio`.

Those are the two features that share `trend_recent_impr` with the target's construction, and ML-07
section 0 measured their residual artefact at **+0.12** and **+0.11** even on the corrected baseline.
A jump of that size from exactly those two features is what the skill calls *suspiciously perfect*,
and this project has been wrong about the same mechanism three times already.

**So the same null runs again**, now against a trained model rather than a correlation. Each page's
future is replaced by a random 30-day window from its own pre-decision history, the target and label
are rebuilt from it, and the whole pipeline retrains on that. The features never change.

A model learning something real about the future should collapse toward the base rate. A model
learning the arithmetic of its own inputs will score just as well, because the arithmetic survives.

**One fix to the baseline row while we are here.** ML-07 quoted the random bar as 0.5432; this
notebook gets 0.5127. Both are single draws, and a single shuffle per seed is itself noisy. The
random baseline is re-run over 20 draws per seed below so the bar stops moving.

In [5]:
# --- a stabler random bar: 20 shuffles per seed, not one
rand_rows = []
for seed in range(10):
    _, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                 .split(ev, groups=ev["client_hash_id"]))
    test = ev.iloc[te]
    draws = [precision_at_k_per_client(test.assign(z=np.random.default_rng(1000 * seed + d)
                                                   .random(len(test))), "z")
             for d in range(20)]
    rand_rows.append({"seed": seed, "random_mean": np.mean(draws), "random_sd": np.std(draws)})
rb = pd.DataFrame(rand_rows)
print(f"random bar over 20 draws x 10 seeds: mean {rb.random_mean.mean():.4f} "
      f"| per-seed sd {rb.random_sd.mean():.4f}")

# --- the null: each page's future replaced by a window from its own past
q_daily = f"""
SELECT content_hash_id, report_date, gsc_impressions FROM {REL}
WHERE report_date < DATE '{D1}' ORDER BY 1, 2"""
daily = con.sql(q_daily).df()
daily = daily[daily["content_hash_id"].isin(set(ev["content_hash_id"]))]
piv = daily.pivot(index="report_date", columns="content_hash_id",
                  values="gsc_impressions").fillna(0)
piv.index = pd.to_datetime(piv.index)
pre_arr = piv.values
print(f"daily history for the null: {piv.shape[1]:,} pages x {piv.shape[0]} days")

rng_null = np.random.default_rng(SEED)
starts = rng_null.integers(0, len(pre_arr) - 30, size=piv.shape[1])
null_future = pd.Series(
    {cid: pre_arr[s:s + 30, i].mean() for i, (cid, s) in enumerate(zip(piv.columns, starts))})

nl = ev.copy()
nl["future_daily_null"] = nl["content_hash_id"].map(null_future)
nl = nl[nl["future_daily_null"].notna()].copy()
nl["target"] = np.arcsinh(nl["future_daily_null"]) - np.arcsinh(nl["baseline_daily"])
nl["declined"] = nl["target"] < 0
print(f"null pool: {len(nl):,} pages | decline rate {nl['declined'].mean():.4f} "
      f"(real: {ev['declined'].mean():.4f})")

null_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(nl, groups=nl["client_hash_id"]))
    train, test = nl.iloc[tr], nl.iloc[te]
    null_rows.append({"seed": seed, "model": "random order",
                      "p_at_100": precision_at_k_per_client(
                          test.assign(z=rng.random(len(test))), "z")})
    for name, feats, kind, make in MODELS:
        Xtr, Xte = train[feats], test[feats]
        if name.startswith(("ridge", "logistic")):
            sc = StandardScaler().fit(Xtr)
            Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
        m = make()
        if kind == "reg":
            m.fit(Xtr, train["target"]); score, asc = m.predict(Xte), True
        else:
            m.fit(Xtr, train["declined"]); score, asc = m.predict_proba(Xte)[:, 1], False
        null_rows.append({"seed": seed, "model": name,
                          "p_at_100": precision_at_k_per_client(
                              test.assign(s=score), "s", ascending=asc)})

nres = pd.DataFrame(null_rows).groupby("model")["p_at_100"].mean().round(4)
side = pd.DataFrame({"real future": tbl["mean"], "null future": nres})
side = side.dropna()
side["real minus null"] = (side["real future"] - side["null future"]).round(4)
side["null base rate"] = round(nl["declined"].mean(), 4)
print()
print(side.sort_values("real future", ascending=False).to_string())

# Each side must be judged against ITS OWN base rate: the null pool's decline
# rate is not the real pool's, so comparing raw scores would flatter one.
real_br, null_br = ev["declined"].mean(), nl["declined"].mean()
share = pd.DataFrame({
    "lift on real future": side["real future"] - real_br,
    "lift on null future": side["null future"] - null_br,
})
share["artefact share"] = (share["lift on null future"] / share["lift on real future"]).round(3)
share = share.round(4).sort_values("lift on real future", ascending=False)
print()
print(f"real base rate {real_br:.4f} | null base rate {null_br:.4f}")
print(share.to_string())

share["signal after removing the artefact"] = (
    share["lift on real future"] - share["lift on null future"]).round(4)
print()
print(share[["lift on real future", "lift on null future",
             "signal after removing the artefact"]].to_string())

# Both baselines restated against the 20-draw bar, since the single-draw
# version in the table above is itself noisy.
stable_bar = rb.random_mean.mean()
print()
print("both baselines against the stabler bar:")
for m in ["baseline  gate + random order", "baseline  ML-07 rule (age order)"]:
    print(f"  {m:34s} {tbl.loc[m, 'mean']:.4f}  vs bar {tbl.loc[m, 'mean'] - stable_bar:+.4f}")

random bar over 20 draws x 10 seeds: mean 0.5132 | per-seed sd 0.0178


daily history for the null: 45,095 pages x 120 days


null pool: 45,095 pages | decline rate 0.5224 (real: 0.5131)



               real future  null future  real minus null  null base rate
model                                                                   
ridge FULL          0.7563       0.5681           0.1882          0.5224
logistic FULL       0.7500       0.5718           0.1782          0.5224
gbm_clf FULL        0.7298       0.5650           0.1648          0.5224
gbm_reg FULL        0.7216       0.5543           0.1673          0.5224
gbm_clf SAFE        0.5526       0.5256           0.0270          0.5224
logistic SAFE       0.5466       0.5217           0.0249          0.5224
ridge SAFE          0.5421       0.5350           0.0071          0.5224
gbm_reg SAFE        0.5245       0.5141           0.0104          0.5224

real base rate 0.5131 | null base rate 0.5224
               lift on real future  lift on null future  artefact share
model                                                                  
ridge FULL                  0.2432               0.0457           0.188
logist

**Verdict: the model is real. Roughly four fifths of its gain survives a future carrying no
information.**

| model | `P@100` | lift over its base rate | lift on the null | artefact share |
|---|---|---|---|---|
| **ridge FULL** | **0.7563** | **+0.2432** | +0.0457 | **0.188** |
| logistic FULL | 0.7500 | +0.2369 | +0.0494 | 0.208 |
| gbm_clf FULL | 0.7298 | +0.2167 | +0.0426 | 0.197 |
| gbm_reg FULL | 0.7216 | +0.2085 | +0.0319 | 0.153 |
| ridge SAFE | 0.5421 | +0.0290 | +0.0126 | — |
| **baseline** gate + random order | **0.5132** | — | — | — |
| **baseline** ML-07 rule (age order) | 0.4882 | −0.0250 | — | — |

**Ridge FULL beats the frozen baseline by +0.2432, and +0.1975 of that survives the null.** This is
the first result in the project where a positive finding holds up under the test that broke three
earlier ones.

**It is consistent with everything before it, not in tension.** ML-07 section 0 measured candidate
C's residual artefact share at 0.19–0.25; the model lands at **0.153–0.208**. The same features under
the *old* baseline carried an artefact **4.6 to 14.6 times larger than their signal**. Fixing the
baseline is what made a model possible — the modelling did not rescue a broken target, the target fix
made the modelling worth doing.

**`SAFE` is close to nothing.** +0.0290 for ridge, and its artefact shares (0.434, −0.728) are ratios
on lifts of 0.03 and 0.01 — too small to divide by, and reported only so nobody reads them as
findings. Almost all the value is in `prior_trend` and `peak_ratio`.

**Two corrections to what came before.**

*The bar is 0.5132, not ML-07's 0.5432.* Averaged over 20 shuffles × 10 seeds, per-seed sd 0.0178.
ML-07 quoted a single draw sitting about 1.7 sd high. The frozen gate does not move; only the
estimate of what random ordering achieves inside it.

*The ML-07 rule reproduces as worse than random* — **−0.0250** here against −0.0391 there,
independently recomputed in a different notebook on a different pool. Its ordering was correctly
withdrawn.

**Ridge beating both trees now reads differently.** Before the null it looked like a warning: linear
models win when the relationship is arithmetic. With the null clearing it, the honest reading is that
the relationship really is close to linear — a fact about the problem, and a reason to prefer the
model you can print over the one you cannot.

**Which model to ship — decided later in this notebook, and not the way this section first argued.**

> ⚠️ **Superseded.** This section originally concluded *logistic, not ridge*, on the grounds that
> ridge's `P@100` lead of **0.0063** sat inside seed-to-seed variation. That reasoning used only
> threshold-based metrics. Once the rank-agreement metric ML-03 declared primary was actually
> computed, ridge's lead came to **+0.0150** — more than twice its `P@100` margin — and the choice
> reversed. See the verdict below the metric suite.

Ridge tops the table by **0.0063** on `P@100` and **0.0065** on AUC, both inside seed-to-seed
variation. It pays for that with an output nobody should read: **R² = 0.0026**, meaning the regression
explains essentially none of the target's variance. It orders well and forecasts nothing.

Logistic returns a probability instead — an output that means something if it ever reaches a
specialist's screen, where a predicted magnitude explaining 0.3% of variance would be actively
misleading. That objection survives, and the answer is to ship ridge's **rank** and never its value.

**Neither model produces a binary.** Both emit a score, the queue sorts by it, and the top 100 per
client are taken. The only threshold in the pipeline is the *evaluation* label `declined = target < 0`,
which scores the queue rather than running it.

### The metrics precision alone was hiding

`precision@100` answers "of the pages we picked, how many fell?" — but `w01` set the cost asymmetry
the other way round: a missed decline is unrecoverable, a false flag costs review time. That makes
**recall** the metric the framing actually cares about, and it has not been reported once.

Three more, each answering a question `P@100` cannot:

| metric | question |
|---|---|
| **recall@100** | of every page that fell, how many did the queue reach? |
| **AUC** | does the model order the *whole* pool correctly, not just its top slice? |
| **R²** | for the regressors, how much of the target's variance is explained at all? |
| **P@20 / P@50** | does the advantage hold at the top of the list, where a specialist starts? |

AUC is legitimate here because `declined` is an explicit *evaluation* binarisation, not the training
target — the distinction ML-05's amendment turns on. R² is reported against the continuous target,
which is what the regressors actually fit.

In [6]:
from sklearn.metrics import roc_auc_score, r2_score
from scipy.stats import spearmanr


def rank_agreement(pred, actual, ascending):
    """The metric ML-03 declared primary and this notebook had not computed.

    Spearman between the predicted ordering and the actual signed change. Needs
    no threshold, so unlike AUC it can tell a hard fall from a mild one. Signed
    so that positive always means 'ordered correctly': a regressor predicts the
    target directly, a classifier predicts P(declined), which runs the other way.
    """
    rho = spearmanr(pred, actual).statistic
    return rho if ascending else -rho

def queue_metrics(frame, score, k, ascending):
    """Pooled across clients: every page counts once."""
    hits = picks = 0
    for _, g in frame.groupby("client_hash_id"):
        top = g.sort_values(score, ascending=ascending).head(min(k, len(g)))
        hits += int(top["declined"].sum())
        picks += len(top)
    total_declines = int(frame["declined"].sum())
    return (hits / picks if picks else np.nan,
            hits / total_declines if total_declines else np.nan)


suite = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]

    z = np.random.default_rng(900 + seed).random(len(test))
    p, rc = queue_metrics(test.assign(z=z), "z", 100, False)
    suite.append({"model": "baseline  gate + random order", "p_at_100": p, "recall_at_100": rc,
                  "auc": 0.5, "r2": np.nan,
                  "spearman": rank_agreement(z, test["target"].values, False),
                  "p_at_20": queue_metrics(test.assign(z=z), "z", 20, False)[0]})

    for name, feats, kind, make in MODELS:
        Xtr, Xte = train[feats], test[feats]
        if name.startswith(("ridge", "logistic")):
            sc_ = StandardScaler().fit(Xtr)
            Xtr, Xte = sc_.transform(Xtr), sc_.transform(Xte)
        m = make()
        if kind == "reg":
            m.fit(Xtr, train["target"])
            pred = m.predict(Xte)
            asc, r2 = True, r2_score(test["target"], pred)
            auc = roc_auc_score(test["declined"], -pred)
            rho = rank_agreement(pred, test["target"].values, True)
        else:
            m.fit(Xtr, train["declined"])
            pred = m.predict_proba(Xte)[:, 1]
            asc, r2 = False, np.nan
            auc = roc_auc_score(test["declined"], pred)
            rho = rank_agreement(pred, test["target"].values, False)
        t = test.assign(s=pred)
        p100, rec = queue_metrics(t, "s", 100, asc)
        suite.append({"model": name, "p_at_100": p100, "recall_at_100": rec,
                      "auc": auc, "r2": r2, "spearman": rho,
                      "p_at_20": queue_metrics(t, "s", 20, asc)[0]})

sm = (pd.DataFrame(suite).groupby("model")
      .agg(spearman=("spearman", "mean"), p_at_20=("p_at_20", "mean"),
           p_at_100=("p_at_100", "mean"), recall_at_100=("recall_at_100", "mean"),
           auc=("auc", "mean"), r2=("r2", "mean"))
      .sort_values("spearman", ascending=False).round(4))
print("D1, mean over ten grouped splits:")
print(sm.to_string())
print()
print(f"pool base rate {ev['declined'].mean():.4f} | AUC 0.5 is chance | "
      f"recall ceiling is capacity-bound: a client with more than 100 decliners cannot exceed "
      f"100/its own count")

print()
print("ridge minus logistic (FULL) -- is ridge's edge worth its uninterpretable output?")
for m in ["p_at_100", "recall_at_100", "auc"]:
    print(f"  {m:15s} {sm.loc['ridge FULL', m] - sm.loc['logistic FULL', m]:+.4f}")
print()
print("spearman is the metric ML-03 declared primary: rank agreement between the")
print("predicted ordering and the actual signed change, no threshold anywhere.")
print("It is the only column here that can distinguish a hard fall from a mild one.")

D1, mean over ten grouped splits:
                               spearman  p_at_20  p_at_100  recall_at_100     auc      r2
model                                                                                    
ridge FULL                       0.5302   0.7828    0.7563         0.1293  0.7584  0.0026
logistic FULL                    0.5152   0.7814    0.7500         0.1293  0.7519     NaN
gbm_reg FULL                     0.4945   0.7477    0.7216         0.1182  0.7380 -0.0020
gbm_clf FULL                     0.4843   0.7769    0.7298         0.1266  0.7487     NaN
logistic SAFE                    0.0553   0.5442    0.5466         0.0930  0.5436     NaN
ridge SAFE                       0.0512   0.5363    0.5421         0.0895  0.5405 -0.2959
baseline  gate + random order   -0.0014   0.5094    0.5129         0.0839  0.5000     NaN
gbm_clf SAFE                    -0.0129   0.5600    0.5526         0.0928  0.5294     NaN
gbm_reg SAFE                    -0.0241   0.5121    0.5245        

**The metric ML-03 declared primary, finally computed — and it disagrees with the others twice.**

| model | **spearman** | P@20 | P@100 | recall@100 | AUC | R² |
|---|---|---|---|---|---|---|
| **ridge FULL** | **0.5302** | 0.7828 | 0.7563 | 0.1293 | 0.7584 | 0.0026 |
| logistic FULL | 0.5152 | 0.7814 | 0.7500 | 0.1293 | 0.7519 | — |
| gbm_reg FULL | 0.4945 | 0.7477 | 0.7216 | 0.1182 | 0.7380 | −0.0020 |
| gbm_clf FULL | 0.4843 | 0.7769 | 0.7298 | 0.1266 | 0.7487 | — |
| logistic SAFE | **0.0553** | 0.5442 | 0.5466 | 0.0930 | 0.5436 | — |
| ridge SAFE | **0.0512** | 0.5363 | 0.5421 | 0.0895 | 0.5405 | −0.2959 |
| baseline, random order | **−0.0014** | 0.5094 | 0.5129 | 0.0839 | 0.5000 | — |
| gbm_clf SAFE | **−0.0129** | 0.5600 | 0.5526 | 0.0928 | 0.5294 | — |
| gbm_reg SAFE | −0.0241 | 0.5121 | 0.5245 | 0.0843 | 0.5092 | −0.4393 |

**The baseline reads −0.0014**, which is the sanity check: random order has no rank agreement with the
target, and the metric says so to three decimal places.

**Disagreement 1 — `SAFE` is far weaker than `P@100` implied.** On `P@100` the SAFE models score
0.5421–0.5526 against a 0.5129 bar, which reads as a modest but real contribution. On rank agreement
they score **0.0512 and 0.0553 against FULL's 0.5302** — an order of magnitude apart. `content_age_days`,
`impr_90d` and `avg_position` can nudge the top hundred pages; they cannot order the pool.

**Disagreement 2, and it is the sharper one — `gbm_clf SAFE` has the best SAFE `P@100` (0.5526) and a
negative Spearman (−0.0129).** It orders the whole pool slightly *worse than random* while scoring
best of its group on the queue metric. `P@100` looks at one hundred pages per client; a model can win
there on the top slice and be wrong everywhere else. This is exactly the blindness that made a
threshold-free primary metric worth declaring, and it took until now to demonstrate.

**The shipping recommendation changes to ridge FULL.** Earlier this notebook recommended logistic,
reasoning that the 0.0063 gap on `P@100` was noise and ridge's R² of 0.0026 made its output unfit to
show anyone. The first half no longer holds: ridge leads by **+0.0150** on rank agreement, more than
twice its `P@100` margin, and logistic trains on a binarised label so it discards magnitude by
construction. The second half was always a presentation problem, not a modelling one — **ship the
rank, never the value**, which is what a queue does anyway.

**Also note the tree models swap places.** By AUC, `gbm_clf` beats `gbm_reg`; by rank agreement,
`gbm_reg` beats `gbm_clf`. Each wins on the metric matching what it was trained to optimise, which is
what should happen and is worth seeing rather than assuming.

### The segmented model, measured instead of argued

Twice now the answer to "why only five features" has been a constraint — 30 clients split into
data-rich and data-poor leaves the poor segment too small to hold any out. That was the wrong shape of
answer, and partly wrong on its own terms.

**A segmented system does not need both halves modelled separately.** Data-poor clients keep the
five-feature model, which is already validated for them. The only open question is whether the
**data-rich** clients gain from columns the others lack — and that is a grouped split inside one
subpopulation, on roughly 24 clients. ML-05 ran on five or six test clients; this has more.

**Which columns are worth the test.** GA4 is not: ML-06 Test 9 already ran it *per client* and found a
median ρ of +0.037 and +0.015, with 0 of 13 and 2 of 24 clients above \|ρ\| = 0.2. Segmenting cannot
rescue a feature that fails inside the clients that have it.

The keyword and links columns are different — **they have never been tested at all**, on the grounds
that one content type has none of them, so any cohort-wide correlation would measure content type.
That reasoning justified excluding them from a *shared* model. It never justified leaving them
unmeasured where they exist.

In [7]:
# Built from the schema, not from memory. An earlier version of this cell asked
# for referring_domains and keyword_difficulty; neither exists in dim_content,
# and the query failed loudly rather than silently returning nothing.
ENRICH = {
    "keyword": ["search_volume", "cpc", "competition"],
    "links": ["backlinks"],
    # never audited anywhere in this project until now -- derived from the
    # keyword and URL strings, so plausibly present for every client
    "structural": ["keyword_char_count", "keyword_token_count",
                   "url_char_count", "category_count"],
}
ENRICH = {k: [c for c in v if c in ev.columns] for k, v in ENRICH.items()}

for name, cols in ENRICH.items():
    have = ev[cols].notna().all(axis=1)
    per_client = ev.assign(h=have).groupby("client_hash_id")["h"].mean()
    rich = per_client[per_client > 0.95].index
    sub = ev[ev["client_hash_id"].isin(rich) & have]
    print(f"{name:8s} cols {cols}")
    print(f"         clients with >95% coverage: {len(rich)} of {ev['client_hash_id'].nunique()}"
          f" | pages {len(sub):,} | decline rate {sub['declined'].mean():.4f}")
    for col in cols:
        ok = sub[col].notna()
        print(f"           Spearman({col:20s}, target) = "
              f"{sub.loc[ok, col].corr(sub.loc[ok, 'target'], method='spearman'):+.4f}")

rows_s = []
for name, cols in ENRICH.items():
    have = ev[cols].notna().all(axis=1)
    per_client = ev.assign(h=have).groupby("client_hash_id")["h"].mean()
    rich = per_client[per_client > 0.95].index
    sub = ev[ev["client_hash_id"].isin(rich) & have].copy()
    if sub["client_hash_id"].nunique() < 8:
        print(f"{name}: only {sub['client_hash_id'].nunique()} clients, too few to hold out")
        continue
    for seed in range(10):
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                      .split(sub, groups=sub["client_hash_id"]))
        train, test = sub.iloc[tr], sub.iloc[te]
        for label, feats in [("FULL (shared model)", FULL), (f"FULL + {name}", FULL + cols)]:
            sc_ = StandardScaler().fit(train[feats])
            m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
            pred = m.predict(sc_.transform(test[feats]))
            p, rec = queue_metrics(test.assign(s=pred), "s", 100, True)
            rows_s.append({"segment": name, "features": label, "p_at_100": p,
                           "spearman": rank_agreement(pred, test["target"].values, True),
                           "auc": roc_auc_score(test["declined"], -pred)})

seg = (pd.DataFrame(rows_s).groupby(["segment", "features"])
       .agg(spearman=("spearman", "mean"), p_at_100=("p_at_100", "mean"),
            auc=("auc", "mean")).round(4))
print()
print("inside the data-rich clients only, ten grouped splits:")
print(seg.to_string())
print()
print("what the extra columns buy, inside the clients that have them:")
for name in seg.index.get_level_values(0).unique():
    for m_ in ["spearman", "p_at_100", "auc"]:
        d = (seg.loc[(name, f"FULL + {name}"), m_] - seg.loc[(name, "FULL (shared model)"), m_])
        print(f"  {name:8s} {m_:10s} {d:+.4f}")

# url_char_count correlates -0.2225, stronger than content_age_days (-0.1981)
# which is already in the model. Adding it inside a group of four may have
# buried it, so test every candidate on its own before concluding.
solo = [c for g in ENRICH.values() for c in g if ev[c].notna().mean() > 0.95]
rows_1 = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    for label, feats in [("FULL", FULL)] + [(f"FULL + {c}", FULL + [c]) for c in solo]:
        t2, e2 = train.dropna(subset=feats), test.dropna(subset=feats)
        if e2.empty:
            continue
        sc_ = StandardScaler().fit(t2[feats])
        m = Ridge(alpha=1.0).fit(sc_.transform(t2[feats]), t2["target"])
        pred = m.predict(sc_.transform(e2[feats]))
        p, _ = queue_metrics(e2.assign(s=pred), "s", 100, True)
        rows_1.append({"features": label, "p_at_100": p,
                       "spearman": rank_agreement(pred, e2["target"].values, True)})

one = (pd.DataFrame(rows_1).groupby("features")
       .agg(spearman=("spearman", "mean"), p_at_100=("p_at_100", "mean")).round(4))
base = one.loc["FULL"]
one["d_spearman"] = (one["spearman"] - base["spearman"]).round(4)
one["d_p_at_100"] = (one["p_at_100"] - base["p_at_100"]).round(4)
print()
print("each candidate added on its own, ten grouped splits:")
print(one.sort_values("d_spearman", ascending=False).to_string())

keyword  cols ['search_volume', 'cpc', 'competition']
         clients with >95% coverage: 19 of 30 | pages 35,186 | decline rate 0.4480
           Spearman(search_volume       , target) = +0.0124
           Spearman(cpc                 , target) = +0.0438
           Spearman(competition         , target) = +0.0218
links    cols ['backlinks']
         clients with >95% coverage: 4 of 30 | pages 1,719 | decline rate 0.6347
           Spearman(backlinks           , target) = -0.0498
structural cols ['keyword_char_count', 'keyword_token_count', 'url_char_count', 'category_count']
         clients with >95% coverage: 30 of 30 | pages 45,095 | decline rate 0.5131
           Spearman(keyword_char_count  , target) = +0.0581
           Spearman(keyword_token_count , target) = +0.0414
           Spearman(url_char_count      , target) = -0.2225
           Spearman(category_count      , target) = -0.0253


links: only 4 clients, too few to hold out



inside the data-rich clients only, ten grouped splits:
                                spearman  p_at_100     auc
segment    features                                       
keyword    FULL (shared model)    0.5970    0.7865  0.7908
           FULL + keyword         0.5898    0.7739  0.7856
structural FULL (shared model)    0.5302    0.7563  0.7584
           FULL + structural      0.5022    0.7389  0.7441

what the extra columns buy, inside the clients that have them:
  keyword  spearman   -0.0072
  keyword  p_at_100   -0.0126
  keyword  auc        -0.0052
  structural spearman   -0.0280
  structural p_at_100   -0.0174
  structural auc        -0.0143



each candidate added on its own, ten grouped splits:
                            spearman  p_at_100  d_spearman  d_p_at_100
features                                                              
FULL                          0.5302    0.7563      0.0000      0.0000
FULL + category_count         0.5242    0.7543     -0.0060     -0.0020
FULL + keyword_char_count     0.5226    0.7558     -0.0076     -0.0005
FULL + keyword_token_count    0.5224    0.7546     -0.0078     -0.0017
FULL + url_char_count         0.5117    0.7444     -0.0185     -0.0119


### The last three columns in `dim_content`

`keyword_created_date`, `last_optimized_date` and `optimization_eligible_date` have never been tested
anywhere in this project. Saying they were *probably* the same snapshot leak as `content_updated_date`
is the kind of reasoning that produced this whole detour, so they get the same procedure as everything
else — **timing first, because it disqualifies without needing a model.**

ML-06 Finding 1 established the failure mode: `dim_content` is an export-time snapshot, so
`content_updated_date` sits *after* the decision point for **77.3%** of pages at D1. Any date column
in the same table is a candidate for the same problem, and a date recording *when a page was
optimised* is a particularly dangerous one — if that optimisation happened after the decision point,
the column is describing the intervention the model is supposed to predict the need for.

In [8]:
DATE_COLS = ["keyword_created_date", "last_optimized_date", "optimization_eligible_date"]
DATE_COLS = [c for c in DATE_COLS if c in df.columns]
assert DATE_COLS, "date columns not loaded -- an empty result here is a bug, not a finding"

print("timing check against the decision point, on the full cohort:")
for col in DATE_COLS:
    d = pd.to_datetime(df[col], errors="coerce")
    present = d.notna()
    after = (d > DECISION) & present
    print(f"  {col:28s} present {present.mean():6.1%} | "
          f"AFTER the decision point {after.sum() / max(present.sum(), 1):6.1%} | "
          f"latest {d.max().date() if present.any() else 'n/a'}")

print()
print("availability inside the evaluation pool, per client:")
for col in DATE_COLS:
    per = ev.assign(h=pd.to_datetime(ev[col], errors="coerce").notna()) \
            .groupby("client_hash_id")["h"].mean()
    print(f"  {col:28s} overall {per.mean():6.1%} | "
          f"clients at 0%: {(per < 0.01).sum()} of {len(per)}")

# Only columns that are both present-before-the-decision and widely available
# earn a signal test. The rest are disqualified on timing, like
# content_updated_date was.
usable = []
for col in DATE_COLS:
    d = pd.to_datetime(ev[col], errors="coerce")
    after_share = ((d > DECISION) & d.notna()).sum() / max(d.notna().sum(), 1)
    per = ev.assign(h=d.notna()).groupby("client_hash_id")["h"].mean()
    if after_share < 0.01 and (per < 0.01).sum() == 0 and d.notna().mean() > 0.95:
        ev[f"days_since_{col}"] = (DECISION - d).dt.days
        usable.append(f"days_since_{col}")

print()
print(f"columns surviving the timing and availability gates: {usable}")

if usable:
    for f in usable:
        ok = ev[f].notna()
        print(f"  Spearman({f:38s}, target) = "
              f"{ev.loc[ok, f].corr(ev.loc[ok, 'target'], method='spearman'):+.4f}")

    rows_d = []
    for seed in range(10):
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                      .split(ev, groups=ev["client_hash_id"]))
        train, test = ev.iloc[tr], ev.iloc[te]
        for label, feats in [("FULL", FULL)] + [(f"FULL + {f}", FULL + [f]) for f in usable]:
            t2, e2 = train.dropna(subset=feats), test.dropna(subset=feats)
            if e2.empty:
                continue
            sc_ = StandardScaler().fit(t2[feats])
            m = Ridge(alpha=1.0).fit(sc_.transform(t2[feats]), t2["target"])
            pred = m.predict(sc_.transform(e2[feats]))
            p, _ = queue_metrics(e2.assign(s=pred), "s", 100, True)
            rows_d.append({"features": label, "p_at_100": p,
                           "spearman": rank_agreement(pred, e2["target"].values, True)})

    dd = (pd.DataFrame(rows_d).groupby("features")
          .agg(spearman=("spearman", "mean"), p_at_100=("p_at_100", "mean")).round(4))
    b = dd.loc["FULL"]
    dd["d_spearman"] = (dd["spearman"] - b["spearman"]).round(4)
    dd["d_p_at_100"] = (dd["p_at_100"] - b["p_at_100"]).round(4)
    print()
    print(dd.sort_values("d_spearman", ascending=False).to_string())
else:
    print("  none reach a signal test -- all disqualified on timing or availability")

timing check against the decision point, on the full cohort:
  keyword_created_date         present  87.6% | AFTER the decision point   0.0% | latest 2026-03-29
  last_optimized_date          present  19.6% | AFTER the decision point 100.0% | latest 2026-07-06
  optimization_eligible_date   present  19.6% | AFTER the decision point 100.0% | latest 2026-08-20

availability inside the evaluation pool, per client:
  keyword_created_date         overall  79.1% | clients at 0%: 2 of 30
  last_optimized_date          overall  17.1% | clients at 0%: 20 of 30
  optimization_eligible_date   overall  17.1% | clients at 0%: 20 of 30

columns surviving the timing and availability gates: []
  none reach a signal test -- all disqualified on timing or availability


**Verdict: all three disqualified. The optimisation dates are a worse leak than
`content_updated_date`, and they leak the intervention itself.**

| column | present | **after the decision point** | latest value | clients at 0% |
|---|---|---|---|---|
| `keyword_created_date` | 87.6% | **0.0%** | 2026-03-29 | **2 of 30** |
| `last_optimized_date` | 19.6% | **100.0%** | 2026-07-06 | 20 of 30 |
| `optimization_eligible_date` | 19.6% | **100.0%** | **2026-08-20** | 20 of 30 |

**Both optimisation dates sit after the decision point for every single populated row.** Not 77.3%
like `content_updated_date` — **100.0%**. The field is only ever filled when an optimisation
happened, and every recorded optimisation post-dates D1, so the column *is* the intervention record.

**That makes it the most circular leak in the project.** The queue exists to decide which pages get
optimised. A specialist optimises a page *because* it was declining. A model reading
`days_since_last_optimized` would learn "pages someone already decided to fix" and score them
highly — flawless in backtest, worthless on the day, because at the decision point nobody has decided
yet. It would be measuring the outcome of the process it is meant to feed.

**`optimization_eligible_date` reaches 2026-08-20**, which is ten days beyond the date this analysis
was run. It contains scheduled, forward-dated values — a product projection, not an observation.
ML-05's leakage taxonomy already lists product-decision fields as their own exclusion category, and
this is what that category looks like.

**`keyword_created_date` is the closest call and still fails.** It is clean on timing — **0.0%** after
the decision point, latest value two days before it — but 2 of 30 clients have none of it. That is
the same rule that excluded `main_intent` and `competition_level`, and bending it here because the
column looked promising is exactly the reasoning this notebook has spent its length avoiding.

**With this, every column in `dim_content` is accounted for:** 4 identifiers, 2 pipeline-metadata
fields (`provider_used`, `model_used` — which describe how the record was generated, not the page,
and are excluded by judgement rather than by test), and 20 candidate features, each either in the
model or excluded with a measured reason.

**The count that matters is not five.** It is that 20 candidates were examined and one — `peak_ratio` —
carries the model, with four others earning their place by a margin that would not survive much
scrutiny.

### The categoricals nobody tested

`content_type`, `main_intent` and `competition_level` were used in ML-06 Test 6 as *grouping*
variables — to show that missingness tracks category — and were then never asked the obvious question:
do they predict?

That is an omission rather than a decision. The keyword and backlink columns were excluded because
whole clients and one entire content type lack them; `content_type` itself is not missing that way.

Two things to check before adding anything:

1. **Availability** — a categorical absent for some clients would rank data coverage, exactly like
   GA4 would have.
2. **Whether it survives a grouped split.** A category that predicts *within* the training clients but
   not across held-out ones is client structure wearing a category's name — the failure mode Test 10
   found in ML-06 and permutation importance found again for `content_age_days`.

In [9]:
WANTED = ["content_type", "main_intent", "competition_level"]
absent = [c for c in WANTED if c not in ev.columns]
if absent:
    raise KeyError(f"not loaded from dim_content, so never tested: {absent}. "
                   "An empty result here would look like a finding and would not be one.")
cats = WANTED
print("availability of the categoricals inside the evaluation pool:")
for c in cats:
    by_client = ev.groupby("client_hash_id")[c].apply(lambda s: s.notna().mean() * 100)
    print(f"  {c:20s} overall {ev[c].notna().mean():6.1%} | "
          f"levels {ev[c].nunique()} | per-client min {by_client.min():5.1f}% "
          f"max {by_client.max():5.1f}% | clients at 0%: {int((by_client < 1).sum())}")

usable = [c for c in cats
          if ev.groupby("client_hash_id")[c].apply(lambda s: s.notna().mean()).min() > 0.5]
print()
print(f"usable without ranking availability: {usable}")

if usable:
    enc = pd.get_dummies(ev[usable].fillna("missing"), prefix=usable, drop_first=True)
    CAT = list(enc.columns)
    ev_c = pd.concat([ev.reset_index(drop=True), enc.reset_index(drop=True).astype(float)], axis=1)
    print(f"encoded into {len(CAT)} indicator columns")

    rows_c = []
    for seed in range(10):
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                      .split(ev_c, groups=ev_c["client_hash_id"]))
        train, test = ev_c.iloc[tr], ev_c.iloc[te]
        for label, feats in [("FULL", FULL), ("FULL + categoricals", FULL + CAT),
                             ("categoricals only", CAT)]:
            sc_ = StandardScaler().fit(train[feats])
            m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
            pred = m.predict(sc_.transform(test[feats]))
            p, rec = queue_metrics(test.assign(s=pred), "s", 100, True)
            rows_c.append({"features": label, "p_at_100": p, "recall_at_100": rec,
                           "auc": roc_auc_score(test["declined"], -pred)})

    cc = (pd.DataFrame(rows_c).groupby("features")
          .agg(p_at_100=("p_at_100", "mean"), recall_at_100=("recall_at_100", "mean"),
               auc=("auc", "mean")).round(4))
    print()
    print("ridge, ten grouped splits:")
    print(cc.to_string())
    print()
    print(f"random-order bar: p_at_100 {sm.loc['baseline  gate + random order', 'p_at_100']:.4f}")

    print()
    print("what the categorical adds on top of FULL:")
    for m in ["p_at_100", "recall_at_100", "auc"]:
        d = cc.loc["FULL + categoricals", m] - cc.loc["FULL", m]
        print(f"  {m:15s} {d:+.4f}")

availability of the categoricals inside the evaluation pool:
  content_type         overall 100.0% | levels 3 | per-client min 100.0% max 100.0% | clients at 0%: 0
  main_intent          overall  87.2% | levels 4 | per-client min   0.0% max 100.0% | clients at 0%: 2
  competition_level    overall  86.4% | levels 3 | per-client min   0.0% max 100.0% | clients at 0%: 2

usable without ranking availability: ['content_type']
encoded into 2 indicator columns



ridge, ten grouped splits:
                     p_at_100  recall_at_100     auc
features                                            
FULL                   0.7563         0.1293  0.7584
FULL + categoricals    0.7562         0.1301  0.7611
categoricals only      0.5391         0.1011  0.5463

random-order bar: p_at_100 0.5129

what the categorical adds on top of FULL:
  p_at_100        -0.0001
  recall_at_100   +0.0008
  auc             +0.0027


**Verdict: `content_type` is the only categorical that can be used, and it adds nothing.**

| | overall | levels | per-client min | clients at 0% | usable |
|---|---|---|---|---|---|
| `content_type` | **100.0%** | 3 | **100.0%** | 0 | **yes** |
| `main_intent` | 87.2% | 4 | 0.0% | **2** | no |
| `competition_level` | 86.4% | 3 | 0.0% | **2** | no |

`main_intent` and `competition_level` fail for the same reason GA4 did — two clients have **none** of
them. Including either would rank data coverage rather than page health.

Ridge on ten grouped splits:

| features | `P@100` | recall@100 | AUC |
|---|---|---|---|
| `FULL` | **0.7563** | 0.1293 | 0.7584 |
| `FULL` + `content_type` | 0.7562 | 0.1301 | **0.7611** |
| `content_type` alone | 0.5391 | 0.1011 | 0.5463 |
| random order | 0.5129 | — | 0.5000 |

**Alone it carries a little.** AUC 0.5463 against chance, `P@100` 0.5391 against the 0.5129 bar — real
but small, and about the size `SAFE` managed.

**On top of `FULL` it adds nothing.** `P@100` moves by **−0.0001** and AUC by **+0.0027**, both well
inside seed-to-seed variation. Whatever `content_type` knows about decline, `peak_ratio` already
knows.

**So the answer to "is five features too few" is no, but not for the reason a feature count suggests.**
The model is not short of columns; it is short of *independent* information. A sixth feature that is
100% available, cleanly measured and genuinely categorical still moves nothing, because a page's
traffic relative to its own recent history has already absorbed the signal.

**Correction to my expectation.** I predicted `content_type` would predict within training clients and
fail to transfer, the way `content_age_days` does. It does transfer — the "alone" row holds above
random on held-out clients. It is simply redundant, which is a different and less interesting failure.

**One methodological note kept deliberately.** The first run of this test returned an empty result
because the setup query loaded only `content_created_date` from `dim_content` — the categoricals were
never in the frame. A defensive `if c in ev.columns` turned that bug into output reading *"usable:
[]"*, which looks exactly like a finding. The filter now raises instead. Empty results are the most
dangerous kind, because nothing about them announces that no test ran.

### Does CTR carry information despite being miscalibrated?

ML-06 Test 2 found CTR sitting **7–9x below** the 2.78% in `docs/data-dictionary.md`, and this
notebook excluded it on that basis. That reasoning is incomplete.

**A measurement can be wrong in level and right in order.** If every page's CTR is scaled down by
roughly the same factor, the *ranking* survives, and a model that only ever sorts does not care about
the units. ML-05 measured `Spearman(position, CTR)` at **−0.288** across the cohort and −0.231 above
1,000 impressions — correctly signed and stable, which is what a preserved ordering looks like.

The reference pipeline includes `ctr` in `MODEL_NUMERIC_FEATURES` and never questions it. Excluding
it on a calibration argument alone is the mirror of that mistake: both are decisions taken without a
test. This runs the test.

Also included: `log_impr_90d`, since the reference pipeline log-transforms every count and this
notebook feeds raw `impr_90d` to two linear models.

In [10]:
ev["ctr"] = ev["ctr"].fillna(0)
ev["log_impr_90d"] = np.log1p(ev["impr_90d"])

print(f"ctr in the evaluation pool: median {ev['ctr'].median():.3f}% | "
      f"mean {ev['ctr'].mean():.3f}% | zero for {(ev['ctr'] == 0).mean():.1%} of pages")
print(f"Spearman(ctr, target)        {ev['ctr'].corr(ev['target'], method='spearman'):+.4f}")
print(f"Spearman(avg_position, ctr)  "
      f"{ev['avg_position'].corr(ev['ctr'], method='spearman'):+.4f}")

VARIANTS = {
    "FULL": FULL,
    "FULL + ctr": FULL + ["ctr"],
    "FULL + log_impr": [f for f in FULL if f != "impr_90d"] + ["log_impr_90d"],
    "FULL + ctr + log_impr": [f for f in FULL if f != "impr_90d"] + ["ctr", "log_impr_90d"],
}

rows_v = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    for label, feats in VARIANTS.items():
        for mname, make in [("ridge", lambda: Ridge(alpha=1.0)),
                            ("logistic", lambda: LogisticRegression(max_iter=2000,
                                                                    random_state=SEED))]:
            sc_ = StandardScaler().fit(train[feats])
            Xtr, Xte = sc_.transform(train[feats]), sc_.transform(test[feats])
            m = make()
            if mname == "ridge":
                m.fit(Xtr, train["target"]); pred, asc = m.predict(Xte), True
                auc = roc_auc_score(test["declined"], -pred)
            else:
                m.fit(Xtr, train["declined"]); pred = m.predict_proba(Xte)[:, 1]
                asc, auc = False, roc_auc_score(test["declined"], pred)
            p, rec = queue_metrics(test.assign(s=pred), "s", 100, asc)
            rows_v.append({"features": label, "model": mname,
                           "p_at_100": p, "recall_at_100": rec, "auc": auc})

vv = (pd.DataFrame(rows_v).groupby(["model", "features"])
      .agg(p_at_100=("p_at_100", "mean"), recall_at_100=("recall_at_100", "mean"),
           auc=("auc", "mean")).round(4))
print()
print(vv.to_string())
print()
print("change from adding ctr, per model:")
for mname in ["ridge", "logistic"]:
    for m in ["p_at_100", "recall_at_100", "auc"]:
        d = vv.loc[(mname, "FULL + ctr"), m] - vv.loc[(mname, "FULL"), m]
        print(f"  {mname:9s} {m:15s} {d:+.4f}")

print()
print("change from the log transform, per model:")
for mname in ["ridge", "logistic"]:
    for m in ["p_at_100", "recall_at_100", "auc"]:
        d = vv.loc[(mname, "FULL + log_impr"), m] - vv.loc[(mname, "FULL"), m]
        print(f"  {mname:9s} {m:15s} {d:+.4f}")

ctr in the evaluation pool: median 0.156% | mean 0.597% | zero for 26.2% of pages
Spearman(ctr, target)        -0.0189
Spearman(avg_position, ctr)  -0.3055



                                p_at_100  recall_at_100     auc
model    features                                              
logistic FULL                     0.7500         0.1293  0.7519
         FULL + ctr               0.7472         0.1290  0.7343
         FULL + ctr + log_impr    0.7414         0.1299  0.7710
         FULL + log_impr          0.7419         0.1300  0.7853
ridge    FULL                     0.7563         0.1293  0.7584
         FULL + ctr               0.7539         0.1291  0.7448
         FULL + ctr + log_impr    0.7520         0.1297  0.7650
         FULL + log_impr          0.7536         0.1299  0.7766

change from adding ctr, per model:
  ridge     p_at_100        -0.0024
  ridge     recall_at_100   -0.0002
  ridge     auc             -0.0136
  logistic  p_at_100        -0.0028
  logistic  recall_at_100   -0.0003
  logistic  auc             -0.0176

change from the log transform, per model:
  ridge     p_at_100        -0.0027
  ridge     recall_at_100   

**Verdict: CTR does not help — but the reason is redundancy, not miscalibration.**

| | value |
|---|---|
| CTR in the pool | median **0.156%**, mean 0.597% |
| pages with CTR exactly 0 | **26.2%** |
| `Spearman(ctr, target)` | **−0.0189** |
| `Spearman(avg_position, ctr)` | **−0.3055** |

**The premise held.** CTR's *ordering* against position is real and correctly signed at −0.3055, so
the 7–9x calibration gap does not destroy the information. Excluding it on calibration grounds alone
was not a sound argument.

**The conclusion still stands, for a better reason.** Adding `ctr` makes both models worse:

| | `P@100` | recall@100 | AUC |
|---|---|---|---|
| ridge | −0.0024 | −0.0002 | **−0.0136** |
| logistic | −0.0028 | −0.0003 | **−0.0176** |

`Spearman(ctr, target)` is **−0.0189** — CTR barely relates to what happens next. What it *does*
relate to is position, at −0.3055, and `avg_position` is already in the feature set. So CTR
contributes the same information a second time, plus 26.2% structural zeros, and the models pay for
it in AUC.

**This is the same shape as `prior_trend` and `content_type`.** Three features now — one continuous,
one categorical, one behavioural — have each correlated respectably with something and added nothing,
because `peak_ratio` and `avg_position` had already absorbed it. The model is not short of columns.

**The transform question is separate, and it is a real finding.**

| features | `P@100` | recall@100 | AUC |
|---|---|---|---|
| logistic `FULL` | **0.7500** | 0.1293 | 0.7519 |
| logistic `FULL` + `log_impr` | 0.7419 | 0.1300 | **0.7853** |
| ridge `FULL` | **0.7563** | 0.1293 | 0.7584 |
| ridge `FULL` + `log_impr` | 0.7536 | 0.1299 | **0.7766** |

Replacing raw `impr_90d` with `log1p(impr_90d)` **raises AUC by 0.0334** for logistic and 0.0182 for
ridge — a clearly better ordering of the whole pool — while **lowering `P@100`** slightly.

Both are true, and which matters depends on the product. AUC scores the entire ranking; `P@100`
scores only the hundred pages a specialist opens. The queue is delivered at K = 100, so `P@100` is
the metric with a customer, and the log transform is not adopted. It is recorded because a different
K, or a different use of the score, would reverse that call — and because the reference pipeline
log-transforms every count without noting the trade either way.

### Does the model serve the whole lane, or only decline?

The lane is **Growth / Recovery / Momentum**, and every metric in this notebook and in ML-07 scores
`declined = target < 0`. Decline only.

The continuous target does encode all three, but not by sign alone — sign gives *direction*, and the
prior state gives *which kind*:

| prior state | target < 0 | target > 0 |
|---|---|---|
| was declining | still falling | **recovery** |
| was flat or growing | **decline** | **momentum** |

`prior_trend` is already a feature, so the taxonomy needs no extra model. What it does need is
evidence, because a model that ranks decline well may be useless at ranking growth — nothing tested
so far speaks to it.

**The test.** Same fitted model, same held-out clients. Sort the prediction **ascending** for a
decline queue and **descending** for a growth queue, then split the growth queue on `prior_trend` to
separate recovery from momentum. Four outcomes, one score.

In [11]:
ev["was_declining"] = ev["prior_trend"] < 0
ev["grew"] = ev["target"] > 0

print("the lane as a 2x2, on the evaluation pool:")
grid = (ev.groupby(["was_declining", "grew"]).size().unstack()
        .rename(index={False: "was flat/growing", True: "was declining"},
                columns={False: "target < 0", True: "target > 0"}))
print(grid.to_string())
print()
print(f"  decline  (was flat/growing, fell): {int(grid.loc['was flat/growing', 'target < 0']):,}")
print(f"  recovery (was declining, rose):    {int(grid.loc['was declining', 'target > 0']):,}")
print(f"  momentum (was flat/growing, rose): {int(grid.loc['was flat/growing', 'target > 0']):,}")

def outcome_precision(frame, score, k, ascending, label):
    hits = picks = 0
    for _, g in frame.groupby("client_hash_id"):
        top = g.sort_values(score, ascending=ascending).head(min(k, len(g)))
        hits += int(top[label].sum()); picks += len(top)
    return hits / picks if picks else np.nan


lane_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te].copy()
    sc_ = StandardScaler().fit(train[FULL])
    m = Ridge(alpha=1.0).fit(sc_.transform(train[FULL]), train["target"])
    test["pred"] = m.predict(sc_.transform(test[FULL]))
    z = np.random.default_rng(700 + seed).random(len(test))

    dec = test[~test["was_declining"]]
    rec = test[test["was_declining"]]
    mom = test[~test["was_declining"]]
    for name, frame, label, asc in [
            ("decline  (of not-yet-declining)", dec, "declined", True),
            ("recovery (of already-declining)", rec, "grew", False),
            ("momentum (of not-yet-declining)", mom, "grew", False)]:
        if len(frame) == 0:
            continue
        lane_rows.append({
            "queue": name, "seed": seed,
            "base_rate": frame[label].mean(),
            "model": outcome_precision(frame, "pred", 100, asc, label),
            "random": outcome_precision(frame.assign(z=z[:len(frame)] if False else
                                                     np.random.default_rng(seed).random(len(frame))),
                                        "z", 100, False, label)})

lane = (pd.DataFrame(lane_rows).groupby("queue")
        .agg(base_rate=("base_rate", "mean"), p_at_100=("model", "mean"),
             random=("random", "mean")).round(4))
lane["vs random"] = (lane["p_at_100"] - lane["random"]).round(4)
print()
print("one model, three queues, ten grouped splits:")
print(lane.to_string())

the lane as a 2x2, on the evaluation pool:
grew              target < 0  target > 0
was_declining                           
was flat/growing       11994       17684
was declining          11427        3990

  decline  (was flat/growing, fell): 11,994
  recovery (was declining, rose):    3,990
  momentum (was flat/growing, rose): 17,684



one model, three queues, ten grouped splits:
                                 base_rate  p_at_100  random  vs random
queue                                                                  
decline  (of not-yet-declining)     0.3845    0.6323  0.4212     0.2111
momentum (of not-yet-declining)     0.6087    0.7460  0.5726     0.1734
recovery (of already-declining)     0.2815    0.3395  0.2433     0.0962


**Verdict: one model serves all three queues. The lane was never three problems.**

The evaluation pool as the 2×2:

| | target < 0 | target > 0 |
|---|---|---|
| **was flat or growing** | **decline** 11,994 | **momentum** 17,684 |
| **was declining** | still falling 11,427 | **recovery** 3,990 |

Same fitted ridge, same held-out clients, ten grouped splits — only the sort direction and the
population change:

| queue | base rate | `P@100` | random | vs random |
|---|---|---|---|---|
| **decline** (of not-yet-declining) | 0.3845 | **0.6323** | 0.4212 | **+0.2111** |
| **momentum** (of not-yet-declining) | 0.6087 | **0.7460** | 0.5726 | **+0.1734** |
| **recovery** (of already-declining) | 0.2815 | **0.3395** | 0.2433 | **+0.0962** |

**All three beat random, and this is the strongest justification yet for the continuous target.** The
original design needed three separate models on three separate populations. One regression on one
continuous value, sorted two ways and split on a feature already in the model, does all of it — and
`w02`'s amendment claim that "no information is lost by collapsing them" is now measured rather than
asserted.

**Momentum is nearly as strong as decline** — **+0.1734** against **+0.2111** — which is a product
finding, not just a modelling one. The growth half of the lane works, and nothing in ML-07 or the
rest of ML-08 would have revealed that, because every metric before this one scored decline.

**Recovery is the weakest, and the gate is a likely reason.** ML-07's gate requires `slip <= 0.5`,
which deliberately removes pages that have already lost more than half their traffic — sensible for a
decline queue, and exactly the population a recovery queue wants most. Recovery also has the smallest
support at 3,990 pages and the lowest base rate at 0.2815. **+0.0962 over random is real but should be
read as a floor**: a gate designed for recovery rather than inherited from decline would likely do
better.

**What this means for the product.** Three queues, one model, three gates — the model is shared and
the gate is what changes per queue. That is a smaller and more honest system than three models, and
it follows directly from the target redesign rather than from anything ML-08 did.

### Closing the audit: the two reference features never tested, and what "stagnant" means now

Twelve of the reference pipeline's eighteen numeric features have a measured reason for exclusion
here. Two do not — they were simply never tried:

- **`log_clicks_90d`** — clicks rather than impressions, so plausibly distinct from `impr_90d`
- **`days_with_impressions`** — on how many of the 90 days the page appeared at all, a *consistency*
  measure rather than a volume one, and unlike GA4 or keyword data it is available for every client

Describing the feature set as audited while two columns sat untested was overstating it.

**And a claim from ML-02 that was never checked.** Rewriting that notebook, the old "stagnant pages
are a useful negative class" became: *a page that does not move scores near zero and lands in the
middle of the ranking on its own*. That is a testable assertion about this model, and asserting it
was not the same as showing it.

In [12]:
ev["log_clicks_90d"] = np.log1p(ev["clicks_90d"])
ev["days_with_impressions"] = ev["days_with_impressions"].fillna(0)

for f in ["log_clicks_90d", "days_with_impressions"]:
    print(f"Spearman({f:22s}, target) = "
          f"{ev[f].corr(ev['target'], method='spearman'):+.4f}")

EXTRA = {"FULL": FULL,
         "FULL + log_clicks": FULL + ["log_clicks_90d"],
         "FULL + days_with_impr": FULL + ["days_with_impressions"],
         "FULL + both": FULL + ["log_clicks_90d", "days_with_impressions"]}

rows_e, stag_rows = [], []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te].copy()
    for label, feats in EXTRA.items():
        sc_ = StandardScaler().fit(train[feats])
        m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
        pred = m.predict(sc_.transform(test[feats]))
        p, rec = queue_metrics(test.assign(s=pred), "s", 100, True)
        rows_e.append({"features": label, "p_at_100": p, "recall_at_100": rec,
                       "auc": roc_auc_score(test["declined"], -pred)})

    # --- the stagnant claim: do near-zero-target pages sit mid-ranking?
    sc_ = StandardScaler().fit(train[FULL])
    m = Ridge(alpha=1.0).fit(sc_.transform(train[FULL]), train["target"])
    test["pred"] = m.predict(sc_.transform(test[FULL]))
    test["pctile"] = test.groupby("client_hash_id")["pred"].rank(pct=True)
    # SIGNED bands. The unsigned version cannot discriminate: |target| collapses
    # direction, so a big fall lands near percentile 0 and a big rise near 1,
    # and they average to the middle exactly as a stagnant page does.
    bands = [(-1e9, -0.25, "1 large fall"),
             (-0.25, -0.05, "2 small fall"),
             (-0.05, 0.05, "3 STAGNANT"),
             (0.05, 0.25, "4 small rise"),
             (0.25, 1e9, "5 large rise")]
    for lo, hi, name in bands:
        band = test[(test["target"] > lo) & (test["target"] <= hi)]
        if len(band):
            stag_rows.append({
                "band": name, "n": len(band),
                "mean_percentile": band["pctile"].mean(),
                "share_in_top_decile": (band["pctile"] <= 0.10).mean(),
                "share_in_bottom_decile": (band["pctile"] >= 0.90).mean(),
                "share_at_either_extreme": ((band["pctile"] <= 0.10) |
                                            (band["pctile"] >= 0.90)).mean()})

ee = (pd.DataFrame(rows_e).groupby("features")
      .agg(p_at_100=("p_at_100", "mean"), recall_at_100=("recall_at_100", "mean"),
           auc=("auc", "mean")).round(4))
print()
print(ee.to_string())
print()
print("change on top of FULL:")
for lab in ["FULL + log_clicks", "FULL + days_with_impr", "FULL + both"]:
    for m_ in ["p_at_100", "auc"]:
        print(f"  {lab:24s} {m_:10s} {ee.loc[lab, m_] - ee.loc['FULL', m_]:+.4f}")

st = (pd.DataFrame(stag_rows).groupby("band")
      .agg(n=("n", "mean"), mean_percentile=("mean_percentile", "mean"),
           top_decile=("share_in_top_decile", "mean"),
           bottom_decile=("share_in_bottom_decile", "mean"),
           either_extreme=("share_at_either_extreme", "mean")).round(4))
print()
print("where the model ranks pages by what actually happened to them")
print("(percentile 0 = predicted worst decline, 1 = predicted best growth)")
print("top_decile = ranked in the worst 10%, bottom_decile = the best 10%")
print(st.to_string())
print()
print("The claim under test, from ML-02: a stagnant page 'scores near zero and")
print("lands in the middle of the ranking on its own'. It holds only if STAGNANT")
print("sits near 0.5 AND avoids both extremes more than the movers do.")
print(f"  stagnant at either extreme: {st.loc['3 STAGNANT', 'either_extreme']:.4f}")
print(f"  large fall at either extreme: {st.loc['1 large fall', 'either_extreme']:.4f}")
print(f"  large rise at either extreme: {st.loc['5 large rise', 'either_extreme']:.4f}")

Spearman(log_clicks_90d        , target) = -0.0153
Spearman(days_with_impressions , target) = -0.1430



                       p_at_100  recall_at_100     auc
features                                              
FULL                     0.7563         0.1293  0.7584
FULL + both              0.7484         0.1301  0.7783
FULL + days_with_impr    0.7520         0.1305  0.7661
FULL + log_clicks        0.7473         0.1290  0.7808

change on top of FULL:
  FULL + log_clicks        p_at_100   -0.0090
  FULL + log_clicks        auc        +0.0224
  FULL + days_with_impr    p_at_100   -0.0043
  FULL + days_with_impr    auc        +0.0077
  FULL + both              p_at_100   -0.0079
  FULL + both              auc        +0.0199

where the model ranks pages by what actually happened to them
(percentile 0 = predicted worst decline, 1 = predicted best growth)
top_decile = ranked in the worst 10%, bottom_decile = the best 10%
                   n  mean_percentile  top_decile  bottom_decile  either_extreme
band                                                                            
1 large f

**Verdict: the ML-02 claim holds — and the ordering is monotonic across the whole signed range,
which nothing before this had checked.**

| band | n | mean percentile | in predicted-worst 10% | in predicted-best 10% | either extreme |
|---|---|---|---|---|---|
| large fall | 2,910 | **0.3330** | **0.2289** | 0.0253 | 0.2542 |
| small fall | 1,200 | 0.4221 | 0.1269 | 0.0474 | 0.1743 |
| **stagnant** | 891 | **0.5012** | 0.0798 | 0.0900 | **0.1697** |
| small rise | 856 | 0.5153 | 0.0763 | 0.0837 | 0.1600 |
| large rise | 3,494 | **0.6648** | 0.0167 | **0.2081** | 0.2248 |

**The claim survives on both conditions.** Stagnant pages sit at **0.5012** — the middle to three
decimal places — and reach an extreme **16.97%** of the time against **25.42%** for large fallers and
**22.48%** for large risers. They are not merely averaging to the centre; they genuinely avoid the
ends. The unsigned version could not have shown this, because `|target|` collapses direction and
every band averages to 0.5 by symmetry.

**The monotonicity is the bigger finding.** Mean percentile runs **0.3330 → 0.4221 → 0.5012 → 0.5153
→ 0.6648**, strictly increasing across all five bands. **Every metric in this notebook before now
binarised at `target < 0`** — `P@100`, recall and AUC are all blind to whether a page predicted to
fall hard actually fell harder than one predicted to fall mildly. This is the first evidence that the
model orders the *continuous* outcome rather than just sorting decliners from non-decliners.

**The extremes separate cleanly in both directions.** A page that fell hard is **9x** more likely to
sit in the predicted-worst decile than the predicted-best (0.2289 against 0.0253). A page that rose
hard is **12x** more likely to sit in the predicted-best (0.2081 against 0.0167). That is the
behaviour the momentum queue depends on, measured directly rather than inferred from `P@100`.

**Recorded because the first version of this test was wrong.** It banded on `|target|` and returned
0.5012 for stagnant pages and 0.5077 for large movers — which reads as confirmation and is not,
since both are what symmetry produces. A test that cannot fail is not evidence, and the difference
between the two versions is one line.

## 4. Errors and interpretation

A metric without error analysis is decoration, and a +0.24 result is exactly when skipping it is most
tempting. Three questions: what does the model lean on, where is it most wrong, and what do its
failures look like up close.

In [13]:
from sklearn.inspection import permutation_importance

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
              .split(ev, groups=ev["client_hash_id"]))
train, test = ev.iloc[tr], ev.iloc[te]
sc = StandardScaler().fit(train[FULL])
Xtr, Xte = sc.transform(train[FULL]), sc.transform(test[FULL])
best = Ridge(alpha=1.0).fit(Xtr, train["target"])

print("ridge FULL coefficients (standardised, so directly comparable):")
for f, w in sorted(zip(FULL, best.coef_), key=lambda kv: -abs(kv[1])):
    print(f"  {f:18s} {w:+.4f}")

perm = permutation_importance(best, Xte, test["target"], n_repeats=10,
                              random_state=SEED, scoring="r2")
print()
print("permutation importance on held-out clients (drop in R^2 when shuffled):")
for f, m, s in sorted(zip(FULL, perm.importances_mean, perm.importances_std),
                      key=lambda kv: -kv[1]):
    print(f"  {f:18s} {m:+.4f} +/- {s:.4f}")

test = test.assign(pred=best.predict(Xte))
test["picked"] = False
for cid, g in test.groupby("client_hash_id"):
    test.loc[g.nsmallest(min(K, len(g)), "pred").index, "picked"] = True
picked = test[test["picked"]]
print()
print(f"held-out seed 0: {len(test):,} pages, {test['client_hash_id'].nunique()} clients, "
      f"{len(picked):,} picked | precision {picked['declined'].mean():.4f}")

print()
print("where the picks go wrong -- false picks by decile of predicted target:")
picked = picked.assign(band=pd.qcut(picked["pred"], 5, duplicates="drop"))
print(picked.groupby("band", observed=True)
      .agg(n=("declined", "size"), precision=("declined", "mean"))
      .round(3).to_string())

print()
print("precision by client (held-out seed 0):")
byc = (picked.groupby("client_hash_id")
       .agg(picks=("declined", "size"), precision=("declined", "mean"))
       .sort_values("precision").round(3))
print(byc.to_string())

# Three concrete failures: pages the model put near the top of a queue that
# then grew. Shown with the inputs, so the reason is visible rather than asserted.
wrong = (picked[~picked["declined"]]
         .nsmallest(3, "pred")
         [["client_hash_id", "pred", "target", "peak_ratio", "prior_trend",
           "content_age_days", "impr_90d", "avg_position",
           "baseline_daily", "recent_daily", "future_daily"]])
print()
print("three confident picks that grew instead:")
print(wrong.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

print()
print("how often a confident pick grows, by peak_ratio band:")
pb = picked.assign(band=pd.cut(picked["peak_ratio"], [0, 0.5, 0.75, 1.0, 1.5, 99],
                               labels=["<0.5", "0.5-0.75", "0.75-1.0", "1.0-1.5", ">1.5"]))
print(pb.groupby("band", observed=True)
      .agg(n=("declined", "size"), precision=("declined", "mean"))
      .round(3).to_string())

ridge FULL coefficients (standardised, so directly comparable):
  peak_ratio         +0.5578
  content_age_days   -0.1467
  impr_90d           +0.0571
  prior_trend        +0.0406
  avg_position       +0.0340

permutation importance on held-out clients (drop in R^2 when shuffled):
  peak_ratio         +0.6034 +/- 0.0054
  avg_position       +0.0073 +/- 0.0004
  impr_90d           +0.0061 +/- 0.0016
  prior_trend        +0.0038 +/- 0.0005
  content_age_days   -0.0645 +/- 0.0022

held-out seed 0: 14,814 pages, 6 clients, 383 picked | precision 0.8251

where the picks go wrong -- false picks by decile of predicted target:
                   n  precision
band                           
(-1.166, -1.011]  77      0.896
(-1.011, -0.934]  76      0.908
(-0.934, -0.636]  77      0.922
(-0.636, -0.511]  76      0.961
(-0.511, 1.438]   77      0.442

precision by client (held-out seed 0):


                         picks  precision
client_hash_id                           
client_d211cb07b9059bab     72      0.458
client_0e1acc6cd57b0eba      6      0.500
client_599043c0ff13edea      5      0.600
client_73cda7b4e4f265ea    100      0.900
client_e547b89c05043229    100      0.900
client_fef1a8f436438636    100      0.970

three confident picks that grew instead:
         client_hash_id   pred  target  peak_ratio  prior_trend  content_age_days  impr_90d  avg_position  baseline_daily  recent_daily  future_daily
client_e547b89c05043229 -1.142   0.224       0.619       -0.505               467 2,404.000         6.777          31.800        16.533        39.800
client_e547b89c05043229 -1.111   0.330       0.651       -0.695               467 2,799.000         6.476          36.533        20.233        50.800
client_e547b89c05043229 -1.103   0.176       0.662       -0.647               467 2,461.000         5.693          31.967        18.100        38.133

how often a confident

## 5. The touched-once test — a second decision point

Everything above happens at one decision point, and every number in it was used to make a choice:
eight models were compared on the same ten splits and one was named best. That selection happened on
the evaluation data, so 0.7563 is a *selection* figure, not a clean estimate.

A conventional 70/15/15 does not fix this here. The unit that must not leak is the **client**, and
there are 30 of them in the pool — a 15% test set is four or five clients, on a pool whose decline
rate ML-07 measured swinging from **0.3180 to 0.7435** depending which clients were held out. That
number would report the luck of one partition.

**D2 — 2026-05-31 — is a genuinely untouched set.** A different month, no model here has seen it, and
nothing was chosen against it. It also closes the gap the split design leaves open: repeated grouped
CV shows transfer to unseen *clients*, never to a different *period*.

**The protocol, fixed before the numbers are looked at:**

1. Fit on **all** of D1's evaluation pool. No splits — D2 is the held-out set.
2. Rebuild D2's cohort, target and gate with the identical code. No refitting, no retuning.
3. Score once. Report whatever comes back.

ML-06 measured the base rate moving from 40.3% to 36.6% between these dates under the old label, so a
drop here would not be surprising. It would mean the model is period-specific, which is worth knowing
before anyone ships it.

In [14]:
D2 = "2026-05-31"

q2 = f"""
WITH prior AS (
  SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
    SUM(gsc_impressions) AS impr_90d,
    SUM(gsc_sum_position) AS sum_position_90d,
    SUM(gsc_impressions) FILTER (
        WHERE report_date < DATE '{D2}' - INTERVAL 30 DAY) AS older60_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D2}' - INTERVAL 30 DAY) AS recent30_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D2}' - INTERVAL 60 DAY
          AND report_date <  DATE '{D2}' - INTERVAL 30 DAY) AS base30_impr
  FROM {REL}
  WHERE report_date >= DATE '{D2}' - INTERVAL 90 DAY AND report_date < DATE '{D2}'
  GROUP BY content_hash_id HAVING SUM(gsc_impressions) > 0),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS future_impr FROM {REL}
  WHERE report_date >= DATE '{D2}' AND report_date < DATE '{D2}' + INTERVAL 30 DAY
  GROUP BY content_hash_id)
SELECT p.*, COALESCE(f.future_impr, 0) AS future_impr
FROM prior p LEFT JOIN future f USING (content_hash_id)
ORDER BY p.content_hash_id"""

d2 = con.sql(q2).df().merge(dim, on="content_hash_id", how="left")
for c in ["older60_impr", "recent30_impr", "base30_impr"]:
    d2[c] = d2[c].fillna(0)
DEC2 = pd.Timestamp(D2)
d2["baseline_daily"] = d2["older60_impr"] / 60
d2["recent_daily"] = d2["recent30_impr"] / 30
d2["future_daily"] = d2["future_impr"] / 30
d2["target"] = np.arcsinh(d2["future_daily"]) - np.arcsinh(d2["baseline_daily"])
d2["declined"] = d2["target"] < 0
d2["avg_position"] = d2["sum_position_90d"] / d2["impr_90d"].replace(0, np.nan) + 1
d2["content_age_days"] = (DEC2 - pd.to_datetime(d2["content_created_date"])).dt.days
d2["slip"] = np.where(d2["baseline_daily"] > 0,
                      (d2["baseline_daily"] - d2["recent_daily"]) / d2["baseline_daily"], np.nan)
d2["peak_ratio"] = np.where(d2["impr_90d"] > 0,
                            d2["recent_daily"] / (d2["impr_90d"] / 90), np.nan)
d2["prior_trend"] = np.where(d2["base30_impr"] > 0,
                             (d2["recent30_impr"] - d2["base30_impr"]) / d2["base30_impr"], np.nan)

cm2 = d2.groupby("client_hash_id")["impr_90d"].transform("median")
gate2 = ((d2["baseline_daily"] > 0) & (d2["content_age_days"] >= MIN_AGE_DAYS)
         & (d2["impr_90d"] >= cm2) & (d2["slip"].fillna(0) <= 0.5))
ev2 = d2[gate2].dropna(subset=FULL + ["target"]).copy()

seen = set(ev["client_hash_id"])
print(f"D2 cohort {len(d2):,} pages | gated+complete {len(ev2):,} | "
      f"{ev2['client_hash_id'].nunique()} clients")
print(f"  clients also present at D1: "
      f"{ev2['client_hash_id'].isin(seen).groupby(ev2['client_hash_id']).first().sum()} "
      f"of {ev2['client_hash_id'].nunique()}")
print(f"  D2 decline rate {ev2['declined'].mean():.4f} (D1 pool: {ev['declined'].mean():.4f})")

FIT = [("ridge FULL", FULL, "reg", lambda: Ridge(alpha=1.0)),
       ("ridge SAFE", SAFE, "reg", lambda: Ridge(alpha=1.0)),
       ("logistic FULL", FULL, "clf", lambda: LogisticRegression(max_iter=2000, random_state=SEED)),
       ("gbm_clf FULL", FULL, "clf", lambda: HistGradientBoostingClassifier(random_state=SEED))]

rows2 = []
draws = [precision_at_k_per_client(
             ev2.assign(z=np.random.default_rng(500 + d).random(len(ev2))), "z")
         for d in range(20)]
rows2.append({"model": "baseline  gate + random order", "p_at_100": float(np.mean(draws))})
rows2.append({"model": "baseline  ML-07 rule (age order)",
              "p_at_100": precision_at_k_per_client(ev2, "content_age_days")})

for name, feats, kind, make in FIT:
    sc = StandardScaler().fit(ev[feats])
    m = make()
    if kind == "reg":
        m.fit(sc.transform(ev[feats]), ev["target"])
        score, asc = m.predict(sc.transform(ev2[feats])), True
    else:
        m.fit(sc.transform(ev[feats]), ev["declined"])
        score, asc = m.predict_proba(sc.transform(ev2[feats]))[:, 1], False
    rows2.append({"model": name,
                  "p_at_100": precision_at_k_per_client(
                      ev2.assign(s=score), "s", ascending=asc)})

t2 = pd.DataFrame(rows2).set_index("model").round(4)
t2["lift over base rate"] = (t2["p_at_100"] - ev2["declined"].mean()).round(4)
t2["D1 (selection)"] = [tbl["mean"].get(m.replace("baseline  ", "baseline  "), np.nan)
                        for m in t2.index]
t2["D2 minus D1"] = (t2["p_at_100"] - t2["D1 (selection)"]).round(4)
print()
print(f"D2 base rate: {ev2['declined'].mean():.4f}")
print(t2.sort_values("p_at_100", ascending=False).to_string())

# The comparison that matters across periods is the gap over random, not the
# raw score -- the two pools have base rates 27 points apart.
d1_rand = tbl.loc["baseline  gate + random order", "mean"]
d2_rand = t2.loc["baseline  gate + random order", "p_at_100"]
print()
for m in ["ridge FULL", "logistic FULL", "gbm_clf FULL", "ridge SAFE"]:
    a1 = tbl.loc[m, "mean"] - d1_rand
    a2 = t2.loc[m, "p_at_100"] - d2_rand
    print(f"  {m:15s} advantage over random: D1 {a1:+.4f} | D2 {a2:+.4f} "
          f"| retained {a2 / a1:.0%}")

D2 cohort 251,951 pages | gated+complete 38,725 | 31 clients
  clients also present at D1: 28 of 31
  D2 decline rate 0.7873 (D1 pool: 0.5131)



D2 base rate: 0.7873
                                  p_at_100  lift over base rate  D1 (selection)  D2 minus D1
model                                                                                       
ridge FULL                          0.9065               0.1192          0.7563       0.1502
logistic FULL                       0.9043               0.1170          0.7500       0.1543
gbm_clf FULL                        0.8872               0.0999          0.7298       0.1574
ridge SAFE                          0.7794              -0.0079          0.5421       0.2373
baseline  gate + random order       0.7774              -0.0099          0.5127       0.2647
baseline  ML-07 rule (age order)    0.7682              -0.0191          0.4882       0.2800

  ridge FULL      advantage over random: D1 +0.2436 | D2 +0.1291 | retained 53%
  logistic FULL   advantage over random: D1 +0.2373 | D2 +0.1269 | retained 53%
  gbm_clf FULL    advantage over random: D1 +0.2171 | D2 +0.1098 | retain

**Verdict: the ordering transfers to an unseen month, at roughly half strength. `SAFE` does not
transfer at all.**

D2: 38,725 gated pages, 31 clients (28 also present at D1), base rate **0.7873** against D1's 0.5131.

| model | D2 `P@100` | lift over D2 base rate | D1 (selection) |
|---|---|---|---|
| **ridge FULL** | **0.9065** | **+0.1192** | 0.7563 |
| logistic FULL | 0.9043 | +0.1170 | 0.7500 |
| gbm_clf FULL | 0.8872 | +0.0999 | 0.7298 |
| ridge SAFE | 0.7794 | **−0.0079** | 0.5421 |
| baseline gate + random | 0.7774 | −0.0099 | 0.5127 |
| baseline ML-07 rule | 0.7682 | **−0.0191** | 0.4882 |

**Read the lift, not the score.** `P@100` rose from 0.7563 to **0.9065**, which looks like the model
improved. It did not. D2's pool declines at **78.7%** against D1's 51.3%, so picking 90.65% out of a
pool where four in five fall anyway is a smaller achievement than picking 75.63% where only half do.
Against random ordering the advantage is **+0.2436 at D1 and +0.1291 at D2** — about **53% retained**.

**That is a pass, and a qualified one.** Nothing was tuned, refit or chosen against D2, and the
ordering still beats random by 0.129 on a month the model never saw. Three independent findings hold
across both dates: the trend features carry the signal, `SAFE` does not, and the ML-07 rule ranks
below random — **−0.0191** here, a third confirmation on a third pool.

**Why the drop is expected rather than alarming.** ML-06 measured the base rate itself moving between
these dates, and ML-07 found the pool's decline rate swinging 0.3180 to 0.7435 across client
partitions. A model whose advantage halves between two periods with base rates 27 points apart is
behaving like a model, not like an artefact — an artefact would have held its apparent strength,
because arithmetic does not care what month it is.

**What this does not test.** 28 of 31 D2 clients were in D1's training data, so this is mostly *same
clients, new month* — the realistic deployment case, and a weaker test than fully unseen clients on a
new period. The repeated grouped CV covers the other axis; neither covers both at once.

**`SAFE` collapsing to −0.0079 settles a question ML-07 left open.** `content_age_days`,
`impr_90d` and `avg_position` carry no transferable ordering signal. Combined with the negative
permutation importance for age, the honest conclusion is that the three universally-available fields
cannot rank pages — and any client missing the trend history cannot be served by this model at all.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — clients appear only as `client_hash_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`

**Checked against the training-honest-models skill:**

- [x] Method chosen to fit the question — ranking, so a continuous score at `precision@K`; both a
      regressor on the target and a classifier on the label, since the target was rebuilt continuous
- [x] The baseline appears in the same table as the model, **computed in this notebook run** — the
      cohort, target and gate are rebuilt here and reproduce ML-07 exactly (202,073 / 46,061 /
      0.4228 / 0.5128)
- [x] Same split, same metric, same pool for every row
- [x] Simple before strong — ridge and logistic before the trees, and ridge won
- [x] Errors read before the score was believed — section 4
- [x] Top 3 features named and explained: `peak_ratio` carries the model almost alone (permutation
      importance 0.6034 against 0.0073 for the next), `avg_position` and `impr_90d` contribute
      marginally, and `content_age_days` is **negative** — shuffling it improves held-out fit
- [x] Seeds fixed (`SEED = 8`, splits 0–9) and library versions printed in section 0

**Limitations, stated rather than hidden:**

- The advantage roughly halves between decision points, +0.2436 to +0.1291 against random.
- One held-out client scored **0.458 across 72 picks** — near coin-flip. The mean hides it, and a
  per-client product is experienced per client.
- Precision is not monotonic in confidence: the most confident band scores 0.896, the fourth 0.961.
- The model is effectively one feature. `prior_trend` adds 0.0038 despite correlating +0.5438,
  because it is collinear with `peak_ratio`.
- `SAFE` does not transfer, so clients without usable trend history cannot be served.
- 28 of 31 D2 clients were seen in training; this tests a new period, not new clients on a new period.